# P7. 추출 프롬프트 v1 + LLM 추출 시험

## 1. 이 노트북에서 하는 일
- 앞서 만든 **블록 8개**를 LLM에 넣습니다.
- LLM은 블록에서 **학교가 하는 행동**만 골라 **정해진 양식**으로 돌려줍니다.
- 돌려준 근거 문장이 **원문에 실제로 있는지** 코드가 확인합니다.
- 결과는 `steps_raw.jsonl`에 저장합니다. 이 파일은 **사람이 검수하기 전** 상태입니다 (`review_status = unreviewed`).

## 2. 핵심 생각 (쉽게)
- **LLM은 찾고, 코드는 확인합니다.** LLM이 근거를 지어내도 코드가 원문 대조에서 걸러 냅니다.
- **학교 행동만 뽑습니다.** 의료기관·보건소·법 조문이 하는 일은 학교 행동이 아닙니다. 예: 제11조의 "의사가 보고한다"는 학교 행동이 아닙니다.
- **단계, 강도, 역할은 원문에 있는 만큼만.** 없으면 `UNSPECIFIED`, `null`을 씁니다.

## 3. 블록 불러오기

In [1]:
import json
from preprocess.prevention_legal.pages import DOCS, excerpt_in_page

BLOCKS = "../../data/interim/prevention_legal/blocks.jsonl"
blocks = [json.loads(line) for line in open(BLOCKS, encoding="utf-8")]
key_of = {doc_id: key for key, (doc_id, _) in DOCS.items()}

print("블록", len(blocks), "개")
for b in blocks:
    print(b["chunk_id"], "|", b["text"][:40].replace("​", ""))


블록 8 개
kdca_measles_2026:p0093:b01 | 1. 교내 **홍역(의심)환자** **[*]** **발생 시 반드시 의료
kdca_measles_2026:p0093:b02 | - 고열, 발진과 함께 기침, 콧물, 결막염 등의 증상이 동반하는 경우
kdca_measles_2026:p0093:b03 | 3. 홍역(의심) 진단을 받지 않았으나 **유사증상자가 2주 이내 2명 
kdca_measles_2026:p0024:b01 | (의사)환자를 진단한 경우
발생
24시간 이내에 관할보건소로 신고
moleg_infectious_disease_act_21521:p0018:b01 | 제11조(의사 등의 신고) ① 의사, 치과의사 또는 한의사는 다음 각 호
moleg_infectious_disease_act_21521:p0018:b02 | ③ 제1항 및 제2항에 따라 보고를 받은 의료기관의 장 및 제16조의2에
moleg_infectious_disease_act_21521:p0019:b01 | 제12조(그 밖의 신고의무자) ① 다음 각 호의 어느 하나에 해당하는 사
moleg_infectious_disease_act_21521:p0019:b02 | 2. 학교, 사회복지시설, 병원, 관공서, 회사, 공연장, 예배장소, 선


## 4. 결과 양식 (스키마 v3.1)
LLM이 돌려줄 한 행동의 칸입니다. 칸마다 **허용된 값**이 정해져 있습니다.

In [2]:
from typing import Literal, Optional
from pydantic import BaseModel, Field

# 스키마 v3.1 허용값 (표를 그대로 옮김)
Stage = Literal["PREVENTION", "RESPONSE_1", "RESPONSE_2", "RESPONSE_3", "RECOVERY",
                "ALL_STAGES", "NATIONAL_CRISIS", "UNSPECIFIED"]
Scope = Literal["COMMON", "DISEASE_GROUP", "NAMED_DISEASES", "UNKNOWN"]
Modality = Literal["MANDATORY", "RECOMMENDED", "DISCRETIONARY", "UNSPECIFIED"]
Basis = Literal["EXPLICIT", "EXAMPLE"]

class SchoolActionCand(BaseModel):
    """학교가 하는 행동 한 개 (스키마 v3.1 SchoolAction 의 추출 후보)."""
    action_text: str = Field(description="학교가 하는 행동을 원문에 가깝게 한 문장으로")
    stage: Stage = Field(description="원문이 이 행동을 배치한 대응 단계. 단계가 없으면 UNSPECIFIED")
    scope: Scope = Field(description="COMMON=공통 절차, NAMED_DISEASES=특정 질병명, DISEASE_GROUP=질병군, UNKNOWN=불명확")
    when_text: str = Field(description="적용 조건을 원문 표현 그대로 (예: 교내 홍역(의심)환자 발생 시)")
    timing_text: Optional[str] = Field(default=None, description="기한이 있으면 원문 표현 그대로 (예: 24시간 이내), 없으면 null")
    modality: Modality = Field(description="원문 어미로 확인되는 강도. 확인 어려우면 UNSPECIFIED. 법적 의무로 단정하지 않음")
    role_name: str = Field(description="행동을 하는 학교 내 역할 이름 (예: 담당교사, 보건교사)")
    basis: Basis = Field(description="역할이 명시인지(EXPLICIT) 예시인지(EXAMPLE). 예시 단서가 있으면 EXAMPLE")
    evidence_quote: str = Field(description="근거가 되는 원문 문장을 그대로 복사 (의역 금지)")

class ExtractionResult(BaseModel):
    actions: list[SchoolActionCand] = Field(description="학교 행동이 없으면 빈 목록")

print("스키마 준비 완료: 필드", list(SchoolActionCand.model_fields))


스키마 준비 완료: 필드 ['action_text', 'stage', 'scope', 'when_text', 'timing_text', 'modality', 'role_name', 'basis', 'evidence_quote']


## 5. 프롬프트 v1 (검토 대상)
프롬프트는 LLM에게 주는 **지시문**입니다. 아래 규칙이 맞는지 한 번 읽어 보세요.
규칙이 바뀌면 `prompt_version`을 올립니다(v2 등).

In [3]:
PROMPT_V1 = """당신은 학교 감염병 대응 지침에서 '학교가 하는 행동'만 찾아 구조화하는 도우미입니다.

규칙 (v1)
1. 학교(교직원: 담당교사, 보건교사, 학교장 등)가 해야 하는 행동만 추출합니다.
2. 의료기관, 보건소, 법 조문이 하는 행동은 추출하지 않습니다. (학교 행동이 아니면 actions 를 비웁니다.)
3. 각 행동에는 근거가 되는 원문 문장을 evidence_quote 에 그대로 복사합니다. 의역하지 않습니다.
4. 원문에 없는 내용은 추가하지 않습니다. 단계·기한·강도가 원문에 없으면 UNSPECIFIED 또는 null 을 씁니다.
5. 예방접종·질병 설명처럼 학교 행동이 없는 문단은 빈 목록을 돌려줍니다.
6. 법률상 의무로 단정하지 않습니다. 지침 문장은 지침 행동으로 취급합니다.
"""
print(len(PROMPT_V1), "자 프롬프트 v1 준비")


393 자 프롬프트 v1 준비


### 🧭 검토 칸 (직접 판단)
- 규칙 1~6 중 **고치고 싶은 규칙**이 있나요? → ____
- 빠진 규칙이 있나요? → ____

## 6. LLM 추출 실행 (블록 8개)

In [4]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)

model = init_chat_model("openai:gpt-6-luna", reasoning_effort="none")
extractor = model.with_structured_output(ExtractionResult)

results = []
for b in blocks:
    try:
        out = extractor.invoke([
            ("system", PROMPT_V1),
            ("human", "문서: " + b["document_id"] + " / " + str(b["pdf_page"]) + "쪽 / 블록 " + b["chunk_id"] + "\n\n원문:\n" + b["text"]),
        ])
        results.append((b, out.actions))
        print(f"{b['chunk_id']} -> 학교 행동 {len(out.actions)}개")
    except Exception as e:
        results.append((b, []))
        print(f"{b['chunk_id']} -> 오류 {type(e).__name__}")


kdca_measles_2026:p0093:b01 -> 학교 행동 1개


kdca_measles_2026:p0093:b02 -> 학교 행동 0개


kdca_measles_2026:p0093:b03 -> 학교 행동 0개


kdca_measles_2026:p0024:b01 -> 학교 행동 0개


moleg_infectious_disease_act_21521:p0018:b01 -> 학교 행동 0개


moleg_infectious_disease_act_21521:p0018:b02 -> 학교 행동 0개


moleg_infectious_disease_act_21521:p0019:b01 -> 학교 행동 0개


moleg_infectious_disease_act_21521:p0019:b02 -> 학교 행동 0개


## 7. 근거 원문 확인

In [5]:
# 검증 1: 근거 문장이 블록 원문(해당 쪽)에 실제로 있는가
# 검증 2: 값이 허용 목록 안에 있는가 (스키마가 이미 강제하지만 한 번 더 표시)
rows = []
for b, actions in results:
    key = key_of[b["document_id"]]
    for a in actions:
        quote_ok = excerpt_in_page(a.evidence_quote, key, b["pdf_page"])
        rows.append({
            "chunk_id": b["chunk_id"],
            "action_text": a.action_text,
            "stage": a.stage, "scope": a.scope, "modality": a.modality,
            "role_name": a.role_name, "basis": a.basis,
            "when_text": a.when_text, "timing_text": a.timing_text,
            "evidence_quote": a.evidence_quote,
            "quote_check": "원문 확인" if quote_ok else "확인 필요",
        })

print("학교 행동 후보:", len(rows), "개")
for r in rows:
    print(f"[{r['quote_check']}] {r['chunk_id']} | {r['action_text'][:45]} | 단계 {r['stage']} | 역할 {r['role_name']} ({r['basis']})")


학교 행동 후보: 1 개
[원문 확인] kdca_measles_2026:p0093:b01 | 홍역(의심)환자가 발생하면 반드시 의료기관에서 진료받도록 안내한다. | 단계 UNSPECIFIED | 역할 UNSPECIFIED (EXPLICIT)


## 8. 저장 (검수 전 상태)

In [6]:
import json, pathlib
OUT_STEPS = pathlib.Path("../../data/interim/prevention_legal/steps_raw.jsonl")
OUT_STEPS.parent.mkdir(parents=True, exist_ok=True)

with OUT_STEPS.open("w", encoding="utf-8") as f:
    for r in rows:
        r2 = dict(r, model="gpt-6-luna", prompt_version="v1", review_status="unreviewed")
        f.write(json.dumps(r2, ensure_ascii=False) + "\n")

ok = sum(1 for r in rows if r["quote_check"] == "원문 확인")
print(f"저장: {OUT_STEPS} | 후보 {len(rows)}개 | 원문 확인 {ok}개 | 확인 필요 {len(rows)-ok}개")


저장: ..\..\data\interim\prevention_legal\steps_raw.jsonl | 후보 1개 | 원문 확인 1개 | 확인 필요 0개


## 9. 사람 검수 체크리스트
추출 결과는 **아직 검수 전**입니다. 아래를 하나씩 보고 기록하세요.
- [ ] 학교 행동이 맞는가? (의료기관·보건소 행동을 잘못 넣지 않았는가)
- [ ] 단계(stage)가 원문 위치와 맞는가?
- [ ] 역할(role_name)과 basis(EXPLICIT/EXAMPLE)가 원문 단서와 맞는가?
- [ ] 기한(timing_text)과 강도(modality)를 원문에 없는 내용으로 만들지 않았는가?
- [ ] 근거 문장(evidence_quote)이 원문과 한 글자씩 맞는가? (코드 확인과 별개로 직접 봄)

검수가 끝나면 `review_status`를 `approved` 또는 `rejected`로 바꿉니다.